[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S05/S05_03_gradiente_regresion.ipynb)

# S05_03 · Descenso del gradiente y regresión lineal desde cero
**Sprint 5 · Matemáticas y estadística en código · ⏱ 90 min**

## Qué vas a aprender
1. Qué es una **derivada** (la pendiente de una curva) y cómo aproximarla con código.
2. Qué es una **función de coste** (el error cuadrático medio) y qué significa "minimizarla".
3. A implementar el **descenso del gradiente** paso a paso y a ver el efecto de la tasa de aprendizaje.
4. A construir una **regresión lineal desde cero** y a comprobar que coincide con scikit-learn.

## Contexto TurisData
El cliente quiere saber cómo cambia la ocupación diaria con la temperatura, los festivos y el precio. Para
entender qué hace realmente un modelo por dentro, lo programamos nosotros con NumPy antes de usar una librería.
Es el mismo mecanismo con el que se entrenan las redes neuronales.

> **Idea central:** un modelo es una fórmula con números ajustables (parámetros). Entrenar es buscar los
> parámetros que hacen que el error sea lo más pequeño posible. El gradiente nos dice hacia dónde mover cada parámetro.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEMILLA = 42

## 1 · La derivada: la pendiente en un punto
Si `f(x)` es una curva, su derivada en `x` dice **cuánto sube o baja** `f` si movemos `x` un poquito.
Si la pendiente es positiva, la curva sube a la derecha; si es negativa, baja; si es 0, estamos en un
mínimo o un máximo. Podemos aproximarla sin fórmulas: `(f(x + h) - f(x - h)) / (2h)` con `h` muy pequeño.

In [ ]:
def f(x):
    return (x - 3) ** 2 + 2       # una parábola con el mínimo en x = 3

h = 1e-6
for x in (0, 3, 5):
    pendiente = (f(x + h) - f(x - h)) / (2 * h)
    print(f"x = {x}: pendiente ≈ {pendiente:.3f}")   # negativa a la izquierda del mínimo, 0 en él, positiva a la derecha

### Ejercicio 1 · Derivada numérica
Escribe la función `derivada_numerica(func, x, h=1e-6)` que devuelva la aproximación anterior para cualquier función.

In [ ]:
# TODO: (func(x + h) - func(x - h)) / (2 * h)
...
assert abs(derivada_numerica(lambda x: x ** 2, 3) - 6) < 1e-4, "La derivada de x^2 en x=3 es 6"
assert abs(derivada_numerica(np.sin, 0) - 1) < 1e-4, "La derivada de sen(x) en 0 es 1"
assert abs(derivada_numerica(f, 3)) < 1e-4, "En el mínimo de la parábola la pendiente es 0"
print("Correcto")

## 2 · Función de coste: el error cuadrático medio (MSE)
Para medir lo mal (o bien) que predice un modelo usamos el **error cuadrático medio**:
`MSE = media((y_real - y_predicha)²)`. Elevar al cuadrado hace que los errores grandes pesen más y que no se
cancelen los positivos con los negativos. Un modelo mejor tiene un MSE más pequeño.

In [ ]:
tabla = np.genfromtxt(dato("ocupacion_diaria.csv"), delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
ocupacion, temperatura, festivo, precio = tabla[:, 0], tabla[:, 1], tabla[:, 2], tabla[:, 3]
print(len(ocupacion), "días")

### Ejercicio 2 · MSE
Escribe `mse(y_real, y_pred)`. Guarda en `mse_ingenuo` el MSE de un modelo "ingenuo" que predice **siempre la
media** de la ocupación (será nuestra referencia a batir).

In [ ]:
# TODO: np.mean((y_real - y_pred) ** 2); el modelo ingenuo es un array con la media repetida (o un escalar)
...
assert mse(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 5.0])) == 4 / 3, "MSE de [1,2,3] frente a [1,2,5] es (0+0+4)/3"
assert np.isclose(mse_ingenuo, ocupacion.var()), "Predecir siempre la media da un MSE igual a la varianza de los datos"
print(f"Correcto. Referencia: MSE ingenuo = {mse_ingenuo:.2f} (error típico ≈ {np.sqrt(mse_ingenuo):.1f} puntos)")

## 3 · Descenso del gradiente en una dimensión
Queremos el `x` que minimiza `f(x)`. Idea: **empezar en cualquier sitio y dar pasos cuesta abajo**.
En cada paso: `x ← x - tasa * pendiente`. La **tasa de aprendizaje** decide el tamaño del paso:
muy pequeña = lento; muy grande = se pasa del mínimo y puede divergir.

In [ ]:
def descender(derivada, x0, tasa, pasos):
    x = x0
    camino = [x]
    for _ in range(pasos):
        x = x - tasa * derivada(x)
        camino.append(x)
    return np.array(camino)

derivada_f = lambda x: 2 * (x - 3)          # derivada exacta de (x - 3)^2 + 2

for tasa in (0.01, 0.1, 1.05):
    camino = descender(derivada_f, x0=-4.0, tasa=tasa, pasos=30)
    print(f"tasa {tasa:>5}: x final = {camino[-1]:.3f}")

### Ejercicio 3 · Tu propio descenso
Usando `descender`, encuentra el mínimo de `g(x) = x² - 6x + 20` (mínimo en `x = 3`). Su derivada es `2x - 6`.
Guarda en `x_minimo` el valor final tras **50 pasos** con tasa `0.1` partiendo de `x0 = 10`.

In [ ]:
# TODO: descender(lambda x: 2 * x - 6, 10.0, 0.1, 50) y toma el último valor
...
assert abs(x_minimo - 3.0) < 0.01, f"Tras 50 pasos debería estar cerca de 3.0 y estás en {x_minimo:.3f}"
print(f"Correcto: x mínimo ≈ {x_minimo:.4f}")

## 4 · Regresión lineal: el modelo y su gradiente
El modelo más sencillo: `ocupación ≈ w0 + w1 · temperatura`. `w0` es el punto de partida y `w1` cuánto cambia
la ocupación por cada grado. Con la matriz `X` (columna de unos + variable) el modelo es `y_pred = X @ w`.

El gradiente del MSE respecto a `w` tiene una fórmula limpia: **`(2 / n) · Xᵀ (X w − y)`**.

**Estandarizamos** la variable (media 0, desviación 1): así las escalas se parecen y el descenso converge bien.
(Sin estandarizar, un paso adecuado para el término independiente es demasiado grande para la pendiente.)

In [ ]:
media_t, desv_t = temperatura.mean(), temperatura.std()
temperatura_z = (temperatura - media_t) / desv_t
X = np.column_stack([np.ones(len(temperatura_z)), temperatura_z])
y = ocupacion
print(X.shape)

### Ejercicio 4 · El gradiente
Escribe `gradiente_mse(X, y, w)` que devuelva el vector gradiente `(2 / n) · Xᵀ (X w − y)`.
El `assert` lo comparará con el gradiente calculado por derivadas numéricas.

In [ ]:
# TODO: errores = X @ w - y; el gradiente es 2 / n multiplicado por X.T @ errores
...
w_prueba = np.array([50.0, 3.0])
numerico = np.array([
    derivada_numerica(lambda a: mse(y, X @ np.array([a, w_prueba[1]])), w_prueba[0]),
    derivada_numerica(lambda b: mse(y, X @ np.array([w_prueba[0], b])), w_prueba[1]),
])
assert gradiente_mse(X, y, w_prueba).shape == (2,), "El gradiente debe tener un valor por parámetro (2)"
assert np.allclose(gradiente_mse(X, y, w_prueba), numerico, rtol=1e-3), f"Tu gradiente {gradiente_mse(X, y, w_prueba)} no coincide con el numérico {numerico}"
print("Correcto: el gradiente analítico coincide con el numérico")

### Ejercicio 5 · Entrenar: el bucle del descenso del gradiente
Escribe `entrenar(X, y, tasa=0.1, epocas=200)` que:
1. Empiece con `w = np.zeros(X.shape[1])`.
2. En cada época: calcule el gradiente y actualice `w = w - tasa * gradiente`.
3. Guarde el MSE de cada época en la lista `historial`.
4. Devuelva `(w, historial)`.

In [ ]:
# TODO: bucle for de epocas: w = w - tasa * gradiente_mse(X, y, w); añade mse(y, X @ w) a historial
...
w_z, historial = entrenar(X, y)
w_exacto = np.linalg.solve(X.T @ X, X.T @ y)
assert len(historial) == 200, "El historial debe tener un MSE por época (200)"
assert historial[-1] < historial[0], "El error debe bajar durante el entrenamiento"
assert all(a >= b - 1e-9 for a, b in zip(historial, historial[1:])), "Con esta tasa el error nunca debería subir"
assert np.allclose(w_z, w_exacto, atol=1e-3), f"Deberías llegar a la solución exacta {w_exacto.round(3)} y estás en {w_z.round(3)}"
print("Correcto:", w_z.round(3), "≈", w_exacto.round(3))

In [ ]:
# Curva de aprendizaje y recta ajustada
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].plot(historial)
ejes[0].set_xlabel("Época")
ejes[0].set_ylabel("MSE")
ejes[0].set_title("El error baja en cada paso")

ejes[1].scatter(temperatura, ocupacion, s=6, alpha=0.4, label="Días")
t_lin = np.linspace(temperatura.min(), temperatura.max(), 50)
ejes[1].plot(t_lin, w_z[0] + w_z[1] * (t_lin - media_t) / desv_t, color="crimson", label="Recta ajustada")
ejes[1].set_xlabel("Temperatura (°C)")
ejes[1].set_ylabel("Ocupación (%)")
ejes[1].set_title("Regresión desde cero: ocupación frente a temperatura")
ejes[1].legend()
plt.tight_layout()
plt.show()

## 5 · La tasa de aprendizaje
Probamos varias tasas con el mismo número de épocas: una muy pequeña avanza lento; una adecuada converge; una
demasiado grande **diverge** (el error crece).

In [ ]:
plt.figure(figsize=(6.5, 3.8))
for tasa in (0.005, 0.1, 0.9):
    _, hist = entrenar(X, y, tasa=tasa, epocas=60)
    plt.plot(hist, label=f"tasa = {tasa}")
plt.yscale("log")
plt.xlabel("Época")
plt.ylabel("MSE (escala logarítmica)")
plt.title("Efecto de la tasa de aprendizaje")
plt.legend()
plt.tight_layout()
plt.show()

### Ejercicio 6 · Diagnosticar la tasa
Entrena con `tasa = 2.5` durante 30 épocas y guarda en `diverge` un booleano que indique si el MSE final es
**mayor** que el MSE inicial (es decir, el entrenamiento se ha descontrolado).

In [ ]:
# TODO: entrenar con tasa=2.5, epocas=30 y compara historial[-1] con historial[0]
...
assert diverge, "Con tasa 2.5 el error debe crecer (divergencia). ¿Usaste tasa=2.5 y 30 épocas?"
print("Correcto: con una tasa demasiado grande el descenso se descontrola")

## 6 · De vuelta a las unidades originales y comparación con scikit-learn
Entrenamos con la temperatura estandarizada, así que `w1` es "puntos de ocupación por cada desviación típica".
Para volver a "puntos por grado": `pendiente = w1 / desv` e `intercepto = w0 − w1 · media / desv`.
Comparamos con `LinearRegression` de scikit-learn.

In [ ]:
from sklearn.linear_model import LinearRegression

pendiente_scratch = w_z[1] / desv_t
intercepto_scratch = w_z[0] - w_z[1] * media_t / desv_t

modelo = LinearRegression().fit(temperatura.reshape(-1, 1), ocupacion)
print(f"Desde cero:   ocupación = {intercepto_scratch:.4f} + ({pendiente_scratch:.4f}) x temperatura")
print(f"scikit-learn: ocupación = {modelo.intercept_:.4f} + ({modelo.coef_[0]:.4f}) x temperatura")

### Ejercicio 7 · Comprobar que coinciden
Guarda en `coinciden` un booleano que indique si la pendiente y el intercepto "desde cero" coinciden con los
de scikit-learn (tolerancia `1e-3`). Usa `np.isclose`.

In [ ]:
# TODO: np.isclose(pendiente_scratch, modelo.coef_[0], atol=1e-3) y lo mismo para el intercepto; combina con and
...
assert coinciden, "Tu regresión desde cero y la de scikit-learn deberían coincidir; revisa el cambio de escala"
print("Correcto: ¡tu descenso del gradiente reproduce a scikit-learn!")

## 7 · Mini-reto integrador: regresión múltiple desde cero
Ahora con **tres variables**: temperatura, festivo y precio medio. La función `entrenar` ya sirve tal cual (el
gradiente es el mismo para cualquier número de columnas).
1. Estandariza las tres variables (una a una o con broadcasting) y construye `X_multi` con la columna de unos.
2. Entrena con `entrenar(X_multi, y, tasa=0.1, epocas=500)`.
3. Guarda en `efecto_festivo` el efecto del festivo **en puntos de ocupación** (deshaz la estandarización:
   coeficiente / desviación de la variable) y en `r2` el R² del modelo (`1 - MSE / varianza de y`).

In [ ]:
# TODO: apila las tres columnas en una matriz, estandariza con axis=0, añade la columna de unos, entrena y divide el coeficiente entre la desviación de esa variable
...
assert X_multi.shape == (1096, 4), f"X_multi debe ser (1096, 4) y es {X_multi.shape}"
assert np.isclose(efecto_festivo, 8.82, atol=0.05), f"El efecto del festivo debería ser ≈ +8.82 y tienes {efecto_festivo:.2f}"
assert 0.5 < r2 < 0.7, f"El R² debería rondar 0.6 y tienes {r2:.3f}"
modelo_multi = LinearRegression().fit(variables, y)
assert np.isclose(efecto_festivo, modelo_multi.coef_[1], atol=1e-2), "Debe coincidir con scikit-learn"
print(f"Correcto: efecto del festivo ≈ {efecto_festivo:+.2f} puntos · R² = {r2:.3f}")

## 8 · Errores comunes
| Síntoma | Causa | Arreglo |
|---|---|---|
| El error sube o da `nan` | Tasa de aprendizaje demasiado grande | Bajarla (÷10) |
| El error baja, pero muy despacio | Tasa demasiado pequeña o variables sin estandarizar | Estandarizar; subir la tasa o las épocas |
| Las pendientes son absurdas en unidades originales | Olvidar deshacer la estandarización | `coef / desviación` |
| El modelo no tiene término independiente | Olvidar la columna de unos | `np.column_stack([np.ones(n), ...])` |
| Signo del gradiente al revés | Escribir `w + tasa * gradiente` | Siempre **restar** el gradiente |

## 9 · Para reflexionar
1. ¿Por qué el gradiente apunta "cuesta arriba" y nosotros lo **restamos**?
2. ¿Qué diferencia hay entre resolver la ecuación normal (cuaderno anterior) y usar el descenso del gradiente? ¿Cuándo preferirías cada uno?
3. Un R² de 0,6 con temperatura, festivo y precio: ¿qué crees que explica el 40 % restante?

## 10 · Qué has aprendido
- La derivada mide la pendiente; el gradiente reúne las pendientes respecto a cada parámetro.
- Entrenar = repetir "calcular el gradiente y dar un paso cuesta abajo".
- La tasa de aprendizaje y la estandarización deciden si el entrenamiento converge.
- Tu regresión desde cero reproduce a scikit-learn: no hay magia.